In [1]:
import polars as pl

In [2]:
data_path = "../data/raw/NAS100_30m.parquet"

In [4]:
df = pl.read_parquet(data_path)
print(df.head(5))

shape: (5, 7)
┌─────────────────────┬─────────┬─────────┬─────────┬─────────┬────────┬────────────┐
│ DateTime            ┆ Open    ┆ High    ┆ Low     ┆ Close   ┆ Volume ┆ TickVolume │
│ ---                 ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---    ┆ ---        │
│ datetime[μs]        ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ i64    ┆ i64        │
╞═════════════════════╪═════════╪═════════╪═════════╪═════════╪════════╪════════════╡
│ 2025-10-01 07:00:00 ┆ 24579.7 ┆ 24589.7 ┆ 24579.0 ┆ 24585.5 ┆ 0      ┆ 911        │
│ 2025-10-01 06:30:00 ┆ 24578.1 ┆ 24582.4 ┆ 24571.5 ┆ 24579.5 ┆ 0      ┆ 1854       │
│ 2025-10-01 06:00:00 ┆ 24595.3 ┆ 24596.1 ┆ 24573.5 ┆ 24578.3 ┆ 0      ┆ 2281       │
│ 2025-10-01 05:30:00 ┆ 24607.5 ┆ 24608.2 ┆ 24591.4 ┆ 24594.7 ┆ 0      ┆ 2010       │
│ 2025-10-01 05:00:00 ┆ 24611.9 ┆ 24613.2 ┆ 24595.1 ┆ 24607.7 ┆ 0      ┆ 2555       │
└─────────────────────┴─────────┴─────────┴─────────┴─────────┴────────┴────────────┘


First, we need to handle time zone alignment and account for Daylight Saving Time (DST) changes.

In [6]:
sort_time_by_volume = (
    df.with_columns(pl.col("DateTime").dt.time().alias("TimeOfDay"))
    .group_by("TimeOfDay")
    .agg(pl.col("TickVolume").mean().alias("MeanTickVolume"))
    .sort("MeanTickVolume", descending=True)
)

print(sort_time_by_volume.head(5))

shape: (5, 2)
┌───────────┬────────────────┐
│ TimeOfDay ┆ MeanTickVolume │
│ ---       ┆ ---            │
│ time      ┆ f64            │
╞═══════════╪════════════════╡
│ 16:30:00  ┆ 11278.058022   │
│ 17:00:00  ┆ 9825.238242    │
│ 17:30:00  ┆ 8866.729231    │
│ 18:00:00  ┆ 8140.576264    │
│ 18:30:00  ┆ 7432.058487    │
└───────────┴────────────────┘


We assume the market open at 9:30 AM will have the highest average trading volume. This insight allows us to infer the dataset's native time zone. Here, the peak volume confirms the data is in EET, meaning we must convert the timestamps accordingly. If your dataset's time zone is already known, you can skip this step.

In [8]:
df = df.with_columns(
    pl.col("DateTime")
    .dt.replace_time_zone("EET")
    .dt.convert_time_zone("America/New_York")
    .alias("DateTime_NY")
)

print(df.select(["DateTime", "DateTime_NY"]).head(5))

shape: (5, 2)
┌─────────────────────┬────────────────────────────────┐
│ DateTime            ┆ DateTime_NY                    │
│ ---                 ┆ ---                            │
│ datetime[μs]        ┆ datetime[μs, America/New_York] │
╞═════════════════════╪════════════════════════════════╡
│ 2025-10-01 07:00:00 ┆ 2025-10-01 00:00:00 EDT        │
│ 2025-10-01 06:30:00 ┆ 2025-09-30 23:30:00 EDT        │
│ 2025-10-01 06:00:00 ┆ 2025-09-30 23:00:00 EDT        │
│ 2025-10-01 05:30:00 ┆ 2025-09-30 22:30:00 EDT        │
│ 2025-10-01 05:00:00 ┆ 2025-09-30 22:00:00 EDT        │
└─────────────────────┴────────────────────────────────┘


In [9]:
dst_test = df.filter(
    (pl.col("DateTime_NY").dt.date() == pl.date(2023, 3, 10))
    | (pl.col("DateTime_NY").dt.date() == pl.date(2023, 3, 15))
).filter(pl.col("DateTime_NY").dt.time() == pl.time(9, 30))

print(dst_test.select(["DateTime", "DateTime_NY"]))

shape: (2, 2)
┌─────────────────────┬────────────────────────────────┐
│ DateTime            ┆ DateTime_NY                    │
│ ---                 ┆ ---                            │
│ datetime[μs]        ┆ datetime[μs, America/New_York] │
╞═════════════════════╪════════════════════════════════╡
│ 2023-03-15 15:30:00 ┆ 2023-03-15 09:30:00 EDT        │
│ 2023-03-10 16:30:00 ┆ 2023-03-10 09:30:00 EST        │
└─────────────────────┴────────────────────────────────┘


As demonstrated, DST shifts are now correctly handled, and the dataset's timestamps are ready for further analysis.